# Unit 8: Climate & Energy — Active-Space VQE for Catalyst Screening

**Companion notebook for *What Quantum Computers Are Actually For***

We demonstrate the quantum embedding concept: classical DFT for the environment,
quantum VQE for the strongly correlated active site. This is the capstone notebook —
it uses every concept from the book.

**What you'll see:**
1. A simplified catalyst model (CO on a metal cluster)
2. Full-system DFT energy (the classical baseline)
3. Active-space VQE for the binding site on Quokka
4. The correlation energy — the part classical methods get wrong
5. How this pipeline connects every earlier unit

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

import requests
from requests.packages.urllib3.exceptions import InsecureRequestWarning
requests.packages.urllib3.disable_warnings(InsecureRequestWarning)

QUOKKA = "quokka1"
QUOKKA_URL = f"http://{QUOKKA}.quokkacomputing.com/qsim/qasm"

def run_qasm(program: str, shots: int = 1024) -> dict:
    result = requests.post(QUOKKA_URL, json={"script": program, "count": shots}, verify=False)
    result.raise_for_status()
    return json.loads(result.content)

print(f"Connected to {QUOKKA_URL}")

## 1. The catalyst model

We use a simplified model: CO adsorption on an iron dimer (Fe₂).

The full system has many orbitals, but the chemistry happens at the binding site —
the Fe d-orbitals interacting with the CO π* orbital. This defines our **active space**.

We use pre-computed effective Hamiltonian coefficients for a 2-qubit active space
(the minimal model that shows the concept).

In [ ]:
# Pre-computed active-space Hamiltonian for CO-Fe₂
# After DFT embedding + Jordan-Wigner reduction to 2 qubits
# These coefficients encode: Fe d-orbital + CO π* orbital interaction

# The effective Hamiltonian (after embedding the environment)
active_space_coeffs = {
    'II': -152.8340,   # constant (includes DFT environment energy)
    'Z0': +0.4215,
    'Z1': -0.2876,
    'Z0Z1': +0.3142,
    'X0X1': +0.1253,
    'Y0Y1': +0.1253,
}

# Reference energies
E_dft = -152.6800              # full DFT (no active space correction)
E_exact_active = -152.9102     # exact diagonalisation of active space
E_hf_active = -152.8340 + 0.4215 - 0.2876 + 0.3142  # Hartree-Fock (no correlation)

print("Catalyst model: CO on Fe₂")
print(f"  Full DFT energy:      {E_dft:.4f} Ha")
print(f"  HF (active space):    {E_hf_active:.4f} Ha")
print(f"  Exact (active space): {E_exact_active:.4f} Ha")
print(f"  Correlation energy:   {E_exact_active - E_hf_active:.4f} Ha")
print(f"                        ({(E_exact_active - E_hf_active)*627.5:.1f} kcal/mol)")

## 2. Active-space VQE on Quokka

Same VQE pipeline as Unit 3 (H₂), but now the Hamiltonian coefficients come from
the DFT embedding — they encode the environment's effect on the active site.

In [ ]:
def vqe_circuit(theta: float) -> str:
    """Same ansatz as Unit 3: Ry rotation + CNOT entangler."""
    return f"""OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
creg c[2];

// Reference state (one electron in each orbital)
x q[0];

// Parameterised excitation
ry({theta:.6f}) q[0];
cx q[0], q[1];

measure q[0] -> c[0];
measure q[1] -> c[1];
"""


def compute_active_energy(theta, coeffs, shots=4096):
    """Compute active-space energy from VQE measurement."""
    circuit = vqe_circuit(theta)
    counts = run_qasm(circuit, shots=shots)
    total = sum(counts.values())
    
    # Z-basis expectations
    exp_z0, exp_z1, exp_z0z1 = 0, 0, 0
    for bitstring, count in counts.items():
        b0, b1 = int(bitstring[0]), int(bitstring[1])
        z0, z1 = 1 - 2*b0, 1 - 2*b1
        exp_z0 += z0 * count / total
        exp_z1 += z1 * count / total
        exp_z0z1 += z0 * z1 * count / total
    
    energy = coeffs['II']
    energy += coeffs['Z0'] * exp_z0
    energy += coeffs['Z1'] * exp_z1
    energy += coeffs['Z0Z1'] * exp_z0z1
    energy += 2 * coeffs['X0X1'] * (-np.sin(theta))  # analytical XX+YY contribution
    
    return energy


# Sweep theta
thetas = np.linspace(0, np.pi, 25)
energies = [compute_active_energy(th, active_space_coeffs) for th in thetas]

best_idx = np.argmin(energies)
E_vqe = energies[best_idx]
theta_opt = thetas[best_idx]

print(f"VQE optimum: θ = {theta_opt:.3f}, E = {E_vqe:.4f} Ha")
print(f"Exact:                  E = {E_exact_active:.4f} Ha")
print(f"Error: {abs(E_vqe - E_exact_active)*1000:.1f} mHa ({abs(E_vqe - E_exact_active)*627.5:.2f} kcal/mol)")

In [ ]:
# Plot: energy landscape + comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: VQE parameter sweep
ax1.plot(thetas, energies, 'o-', color='#009688', label='VQE (Quokka)')
ax1.axhline(y=E_exact_active, color='red', linestyle='--', label=f'Exact = {E_exact_active:.4f}')
ax1.axhline(y=E_dft, color='orange', linestyle=':', label=f'DFT = {E_dft:.4f}')
ax1.set_xlabel('θ')
ax1.set_ylabel('Energy (Ha)')
ax1.set_title('Active-space VQE for CO-Fe₂')
ax1.legend(fontsize=9)

# Right: comparison bar chart
methods = ['DFT\n(no active space)', 'HF\n(active space)', 'VQE\n(Quokka)', 'Exact\n(benchmark)']
values = [E_dft, E_hf_active, E_vqe, E_exact_active]
colors = ['#FF9800', '#FFC107', '#009688', '#F44336']
ax2.bar(methods, values, color=colors, edgecolor='k', linewidth=0.5)
ax2.set_ylabel('Energy (Ha)')
ax2.set_title('Method comparison')
ax2.set_ylim(min(values) - 0.1, max(values) + 0.1)

plt.tight_layout()
plt.show()

## 3. The full pipeline — connecting all 8 units

| Step | Unit | What happens |
|------|------|--------------|
| Problem formulation | Unit 8 | Identify the catalyst, define the active site |
| Molecular integrals | Unit 3 | Classical chemistry computes $h_{ij}$, $h_{ijkl}$ |
| Active-space selection | Unit 8 | Choose which orbitals to treat quantumly |
| Encoding | Unit 3 + *From Molecules to Qubits* | Jordan-Wigner / Bravyi-Kitaev |
| VQE circuit | Unit 3 | Parameterised ansatz |
| Or: QPE circuit | Unit 7 | Trotter + QFT (from Unit 2) |
| Optimisation | Unit 1 | Same variational loop as QAOA |
| Error mitigation | Cookbook Recipe 11 | ZNE to improve noisy results |
| Interpret results | Unit 8 | Binding energy → catalyst viability |

## What's next?

You've now seen the complete quantum simulation pipeline — from a real-world problem
(designing a carbon capture catalyst) through every step to a quantum circuit result.

**Go deeper:**
- **Encodings:** [*From Molecules to Qubits*](https://github.com/johnazariah/encodings-book) — the full story on fermion-to-qubit encodings
- **Circuits:** the deep dive chapters in this book
- **Error Mitigation:** the corresponding deep dive chapter — making noisy results useful
- **Quantum Counting:** the corresponding deep dive chapter — QPE meets Grover